# Dry Gas–Gas Heat Exchanger Test — Simulation & Rating

This notebook uses the **real property providers**
(`DryAirPropertyProvider`, `GasMixturePropertyProvider`) and the **real
solver** (`BareTubeHeatExchanger.solve`, via `.simulate`/`.rate`) — no
notebook-local correlations.

The tester picks the mode in the **Mode Selection** section below:

- **`mode = "simulation"`**: known inlets + flow rates -> achievable outlet
  temperatures (+ duty). `surface_margin` is an *input* (`0.0` = "on the
  nose"; e.g. `0.15` derates `UA` by 15% before computing the result).
  Also contrasts the default mean-property iteration against the
  `iterate=False` inlet-only single pass.
- **`mode = "rating"`**: known/target outlet temperatures (a *closed* heat
  balance) -> `overdesign_factor` / `ua_margin` as *outputs*. Target outlet
  temperatures default to a `surface_margin=0` reference Simulation run (so
  an out-of-the-box run starts at `overdesign_factor ≈ 0`); edit them to
  explore over-/under-surface.

Both modes share the same geometry and `inside`/`outside` side inputs. A
`surface_margin=0` reference Simulation always runs first — it is the
result itself in `mode="simulation"`, and just the default-seeding baseline
in `mode="rating"`.

Scope stays gas-phase / sensible-only: `H2O` in the outside mixture is a
gas-phase component, not condensing moisture. No PsychroLib, no
`MoistAirState`, no dew point, no condensation.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)

Workspace root: C:\Users\pawel\GitHub\kalkalori


In [2]:
from IPython.display import display

from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle

from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.models.heat_balance import BalanceSideSpec

from core.properties import (
    DryAirPropertyProvider,
    GasMixturePropertyProvider,
    GasMixtureSpec,
    DRY_AIR_MOLAR_MASS,
    component_molar_mass,
)

print("Imports completed.")

Imports completed.


## Input Data

In [3]:
T0_C = 273.15
p_atm = 101_325.0

def c_to_k(t_c: float) -> float:
    return t_c + T0_C

def kgh_to_kgs(m: float) -> float:
    return m / 3600.0

# --- Cold dry air, inside tubes ---------------------------------------------
inside_T_in_C = 30.0
inside_m_dot_kg_h = 18_000.0

# --- Hot wet flue-gas mixture, outside tubes --------------------------------
# dry-air basis 79/21 with 120 g H2O / kg dry air, as wet mole fractions.
outside_T_in_C = 326.0
outside_m_dot_kg_h = 28_000.0
outside_gas_mixture_components = {
    "N2": 0.6627,
    "O2": 0.1761,
    "H2O": 0.1612,
}

# Outside-side Euler correlation (pressure drop, always based on V_max):
# - 'zukauskas': recommended default for bare inline/staggered banks; strongest near
#   SL/D = 1.25, 1.5 or 2.0, with interpolation/clamping outside that grid.
# - 'gaddis_gnielinski': alternative for bare banks and useful cross-check; requires
#   >5 rows inline or >5 effective rows staggered, ST/D > 1 and valid pitch geometry.
# - 'esdu': reserved for future finned-bank support; currently not implemented and
#   invalid for this bare-tube notebook.
euler_provider = 'gaddis_gnielinski'
# euler_provider = 'zukauskas'

# --- Bare tube bundle geometry -----------------------------------------------
tube_Do = 38.1e-3
tube_wall = 1.65e-3
tube_Di = tube_Do - 2.0 * tube_wall
tube_length = 2.2
tube_k = 50.0  # W/(m*K), carbon steel

tube = BareTube(
    D_i=tube_Di,
    D_o=tube_Do,
    length_total=tube_length,
    length_effective=tube_length,
    wall_k=tube_k,
)

bundle = TubeBundle(
    tube=tube,
    n_rows=20,
    n_tubes_per_row=40,
    pitch_transverse=48.0e-3,
    pitch_longitudinal=48.0e-3,
    layout="inline",  # "inline", "staggered"
    n_passes_tube=1,
    flow_arrangement="crossflow",  # "crossflow", "counterflow", "cocurrentflow"
)

hx = BareTubeHeatExchanger(bundle)

pd.Series({
    "inside_T_in_C": inside_T_in_C,
    "inside_m_dot_kg_h": inside_m_dot_kg_h,
    "outside_T_in_C": outside_T_in_C,
    "outside_m_dot_kg_h": outside_m_dot_kg_h,
    "euler_provider": euler_provider,
    "tube_Do_mm": tube_Do * 1000.0,
    "tube_Di_mm": tube_Di * 1000.0,
    "n_tubes_total": bundle.n_tubes_total,
    "n_rows": bundle.n_rows,
    "A_inner_m2": bundle.total_inner_area,
    "A_outer_m2": bundle.total_outer_area,
    "flow_arrangement": bundle.flow_arrangement,
}, name="value").to_frame()

,value
inside_T_in_C,30.0
inside_m_dot_kg_h,18000.0
outside_T_in_C,326.0
outside_m_dot_kg_h,28000.0
euler_provider,gaddis_gnielinski
tube_Do_mm,38.1
tube_Di_mm,34.8
n_tubes_total,800
n_rows,20
A_inner_m2,192.416267


## Mode Selection

Choose what this notebook computes, then run the rest of the notebook top
to bottom.

In [4]:
mode = "rating"  # "simulation" or "rating"
assert mode in ("simulation", "rating")

# --- used only when mode == "simulation" ------------------------------------
# 0.0 = "on the nose" (no margin); e.g. 0.15 derates UA by 15% before
# computing the achievable duty / outlet temperatures.
surface_margin = 0.0

# --- used only when mode == "rating" -----------------------------------------
# Target/measured outlet temperatures that close the heat balance. Left as
# None here; the reference Simulation cell below fills these in from its own
# surface_margin=0 result if they are still None, so a fresh run of this
# notebook with mode="rating" starts at overdesign_factor ~= 0. Set explicit
# values here (and re-run from this cell down) to explore over-/under-surface.
inside_T_out_target_C = 145.0
outside_T_out_target_C = None

print(f"mode = {mode!r}")

mode = 'rating'


## Side Inputs (Used by Both Simulation and Rating)

`HXSideInput` bundles the property provider with the boundary
conditions for one side. "inside" is the tube side, "outside" is the
bundle side; which side is thermally hot vs. cold is decided from `T_in`.
In `mode="rating"`, the same `provider`/`p`/`m_dot`/`T_in` feed into a
`BalanceSideSpec` further down, alongside the target outlet temperature.

In [5]:
inside = HXSideInput(
    provider=DryAirPropertyProvider(),
    m_dot=kgh_to_kgs(inside_m_dot_kg_h),
    T_in=c_to_k(inside_T_in_C),
    p=p_atm,
)

outside_gas_spec = GasMixtureSpec(
    components=outside_gas_mixture_components,
    basis="mole",
    backend="HEOS",
    imposed_phase="gas",
)

outside = HXSideInput(
    provider=GasMixturePropertyProvider(outside_gas_spec),
    m_dot=kgh_to_kgs(outside_m_dot_kg_h),
    T_in=c_to_k(outside_T_in_C),
    p=p_atm,
)

inside_molar_mass = DRY_AIR_MOLAR_MASS
outside_mole_fractions = outside_gas_spec.to_mole_fractions()
outside_molar_mass = sum(
    x * component_molar_mass(comp, molar_masses=outside_gas_spec.molar_masses)
    for comp, x in outside_mole_fractions.items()
)

print("Simulation side inputs prepared.")

Simulation side inputs prepared.


## Reference Simulation (Iterative Mean-Property)

Always runs. In `mode="simulation"` this uses the tester's `surface_margin`
and *is* the requested result. In `mode="rating"` it instead runs at
`surface_margin=0.0` purely as a baseline, used only to seed the default
target outlet temperatures for the Rating section below.

In [6]:
res_mean = hx.simulate(
    inside, outside,
    surface_margin=surface_margin if mode == "simulation" else 0.0,
    euler_provider=euler_provider,
)

assert res_mean.converged
assert res_mean.iterations > 1

# Seed Rating's target outlet temperatures from this reference run, if the
# tester hasn't set them explicitly in Mode Selection above.
if inside_T_out_target_C is None:
    inside_T_out_target_C = res_mean.T_out_inside - T0_C
if outside_T_out_target_C is None:
    outside_T_out_target_C = res_mean.T_out_outside - T0_C

if mode == "simulation":
    print(f"mode = {mode!r}, surface_margin = {surface_margin}")

    display(pd.Series({
        "converged": res_mean.converged,
        "iterations": res_mean.iterations,
        "residual_q_rel": res_mean.residual_q_rel,
        "surface_margin": res_mean.surface_margin,
        "q_kW": res_mean.q / 1e3,
        "Q_full_kW": res_mean.Q_full / 1e3,
        "Q_derated_kW": res_mean.Q_derated / 1e3,
        "UA_W_K": res_mean.UA,
        "U_mean_W_m2K": res_mean.U_mean,
        "T_mean_inside_C": res_mean.T_mean_inside - T0_C,
        "T_mean_outside_C": res_mean.T_mean_outside - T0_C,
        "T_out_inside_C": res_mean.T_out_inside - T0_C,
        "T_out_outside_C": res_mean.T_out_outside - T0_C,
        "inside_v_mean_m_s": res_mean.inside_velocity_mean,
        "outside_v_mean_m_s": res_mean.outside_velocity_mean,
        "inside_Re_mean": res_mean.inside_Re_mean,
        "outside_Re_mean": res_mean.outside_Re_mean,
        "inside_Pr_mean": res_mean.inside_Pr_mean,
        "outside_Pr_mean": res_mean.outside_Pr_mean,
        "inside_alfa_mean_W_m2K": res_mean.inside_alfa_mean,
        "outside_alfa_mean_W_m2K": res_mean.outside_alfa_mean,
        "inside_rho_kg_m3": res_mean.inside_props_mean.rho,
        "outside_rho_kg_m3": res_mean.outside_props_mean.rho,
        "inside_mu_Pa_s": res_mean.inside_props_mean.mu,
        "outside_mu_Pa_s": res_mean.outside_props_mean.mu,
        "inside_k_W_mK": res_mean.inside_props_mean.k,
        "outside_k_W_mK": res_mean.outside_props_mean.k,
        "inside_cp_J_kgK": res_mean.inside_props_mean.cp,
        "outside_cp_J_kgK": res_mean.outside_props_mean.cp,
        "inside_molar_mass_kg_mol": inside_molar_mass,
        "outside_molar_mass_kg_mol": outside_molar_mass,
        "inside_Vdot_m3_s": inside.m_dot / res_mean.inside_props_mean.rho,
        "outside_Vdot_m3_s": outside.m_dot / res_mean.outside_props_mean.rho,
        "inside_Vdot_m3_h": 3600.0 * inside.m_dot / res_mean.inside_props_mean.rho,
        "outside_Vdot_m3_h": 3600.0 * outside.m_dot / res_mean.outside_props_mean.rho,
        "A_inner_m2": bundle.total_inner_area,
        "A_outer_m2": bundle.total_outer_area,
        "inside_dp_Pa": res_mean.final_result.tube_side_hydraulic.dp_total,
        "inside_dp_friction_Pa": res_mean.final_result.tube_side_hydraulic.dp_friction,
        "inside_dp_acceleration_Pa": res_mean.final_result.tube_side_hydraulic.dp_acceleration,
        "inside_dp_bundle_Pa": res_mean.final_result.tube_side_hydraulic.dp_tube_bundle,
        "inside_dp_local_Pa": res_mean.inside_dp_local,
        "outside_dp_drag_Pa": res_mean.outside_dp_drag,
        "outside_dp_acceleration_Pa": res_mean.outside_dp_acceleration,
        "outside_dp_local_Pa": res_mean.outside_dp_local,
        "outside_dp_Pa": res_mean.outside_dp_total,
    }, name="value").to_frame())
else:
    # IMPORTANT: this is only a surface_margin=0.0 REFERENCE run used to
    # auto-seed Rating's default target outlet temperatures below. It is
    # NOT the Rating result -- do not read "surface_margin" or "q_kW" here
    # as the answer. See the "Rating: Closed Heat Balance" section further
    # down for overdesign_factor / ua_margin.
    print(
        f"mode = {mode!r} -- the table below is only a surface_margin=0.0 "
        "REFERENCE run (used to auto-seed the Rating target outlet "
        "temperatures if they were left as None). It is NOT the Rating "
        "result. Scroll down to 'Rating: Closed Heat Balance -> Overdesign' "
        "for the actual overdesign_factor / ua_margin."
    )
    display(pd.Series({
        "reference_q_kW": res_mean.q / 1e3,
        "reference_T_out_inside_C": res_mean.T_out_inside - T0_C,
        "reference_T_out_outside_C": res_mean.T_out_outside - T0_C,
        "inside_T_out_target_C (used by Rating)": inside_T_out_target_C,
        "outside_T_out_target_C (used by Rating)": outside_T_out_target_C,
    }, name="value").to_frame())

mode = 'rating' -- the table below is only a surface_margin=0.0 REFERENCE run (used to auto-seed the Rating target outlet temperatures if they were left as None). It is NOT the Rating result. Scroll down to 'Rating: Closed Heat Balance -> Overdesign' for the actual overdesign_factor / ua_margin.


,value
reference_q_kW,704.698386
reference_T_out_inside_C,169.378387
reference_T_out_outside_C,247.247337
inside_T_out_target_C (used by Rating),145.000000
outside_T_out_target_C (used by Rating),247.247337


## Media Properties Table

Snapshot at converged mean-property state (`res_mean`).

In [7]:
media_table = pd.DataFrame([
    {
        "side": "inside (tube side)",
        "fluid": "dry air",
        "rho_kg_m3": res_mean.inside_props_mean.rho,
        "mu_Pa_s": res_mean.inside_props_mean.mu,
        "k_W_mK": res_mean.inside_props_mean.k,
        "cp_J_kgK": res_mean.inside_props_mean.cp,
        "molar_mass_kg_mol": inside_molar_mass,
        "Vdot_m3_s": inside.m_dot / res_mean.inside_props_mean.rho,
        "Vdot_m3_h": 3600.0 * inside.m_dot / res_mean.inside_props_mean.rho,
    },
    {
        "side": "outside (bundle side)",
        "fluid": "gas mixture (N2/O2/H2O)",
        "rho_kg_m3": res_mean.outside_props_mean.rho,
        "mu_Pa_s": res_mean.outside_props_mean.mu,
        "k_W_mK": res_mean.outside_props_mean.k,
        "cp_J_kgK": res_mean.outside_props_mean.cp,
        "molar_mass_kg_mol": outside_molar_mass,
        "Vdot_m3_s": outside.m_dot / res_mean.outside_props_mean.rho,
        "Vdot_m3_h": 3600.0 * outside.m_dot / res_mean.outside_props_mean.rho,
    },
])

media_table

,side,fluid,rho_kg_m3,mu_Pa_s,k_W_mK,cp_J_kgK,molar_mass_kg_mol,Vdot_m3_s,Vdot_m3_h
0,inside (tube side),dry air,0.946689,0.000022,0.031598,1011.201807,0.028965,5.281566,19013.636615
1,outside (bundle side),gas mixture (N2/O2/H2O),0.589904,0.000027,0.042922,1150.489076,0.027104,13.184809,47465.313283


## Comparison: Inlet-Only Single Pass (`iterate=False`)

Only meaningful for `mode="simulation"` (skipped otherwise). Same inputs,
but properties are evaluated once at the inlet state instead of at the
converged mean bulk state. This is the escape hatch, not the default —
useful here only to quantify how much the large temperature span
(30 → 400 °C) moves the result.

In [8]:
if mode == "simulation":
    res_inlet = hx.simulate(
        inside, outside, iterate=False, euler_provider=euler_provider,
    )

    assert res_inlet.converged
    assert res_inlet.iterations == 1

    duty_shift_pct = (res_mean.q - res_inlet.q) / res_inlet.q * 100.0

    comparison = pd.DataFrame([
        {
            "mode": "mean-property (default)",
            "iterations": res_mean.iterations,
            "q_kW": res_mean.q / 1e3,
            "UA_W_K": res_mean.UA,
            "T_out_inside_C": res_mean.T_out_inside - T0_C,
            "T_out_outside_C": res_mean.T_out_outside - T0_C,
            "inside_v_m_s": res_mean.inside_velocity_mean,
            "outside_v_m_s": res_mean.outside_velocity_mean,
            "inside_rho_kg_m3": res_mean.inside_props_mean.rho,
            "outside_rho_kg_m3": res_mean.outside_props_mean.rho,
            "inside_mu_Pa_s": res_mean.inside_props_mean.mu,
            "outside_mu_Pa_s": res_mean.outside_props_mean.mu,
            "inside_k_W_mK": res_mean.inside_props_mean.k,
            "outside_k_W_mK": res_mean.outside_props_mean.k,
            "inside_cp_J_kgK": res_mean.inside_props_mean.cp,
            "outside_cp_J_kgK": res_mean.outside_props_mean.cp,
            "inside_molar_mass_kg_mol": inside_molar_mass,
            "outside_molar_mass_kg_mol": outside_molar_mass,
            "inside_Vdot_m3_s": inside.m_dot / res_mean.inside_props_mean.rho,
            "outside_Vdot_m3_s": outside.m_dot / res_mean.outside_props_mean.rho,
            "A_inner_m2": bundle.total_inner_area,
            "A_outer_m2": bundle.total_outer_area,
            "inside_dp_Pa": res_mean.final_result.tube_side_hydraulic.dp_total,
            "inside_dp_friction_Pa": res_mean.final_result.tube_side_hydraulic.dp_friction,
            "inside_dp_acceleration_Pa": res_mean.final_result.tube_side_hydraulic.dp_acceleration,
            "inside_dp_bundle_Pa": res_mean.final_result.tube_side_hydraulic.dp_tube_bundle,
            "inside_dp_local_Pa": res_mean.inside_dp_local,
            "outside_dp_drag_Pa": res_mean.outside_dp_drag,
            "outside_dp_acceleration_Pa": res_mean.outside_dp_acceleration,
            "outside_dp_local_Pa": res_mean.outside_dp_local,
            "outside_dp_Pa": res_mean.outside_dp_total,
        },
        {
            "mode": "inlet-only (iterate=False)",
            "iterations": res_inlet.iterations,
            "q_kW": res_inlet.q / 1e3,
            "UA_W_K": res_inlet.UA,
            "T_out_inside_C": res_inlet.T_out_inside - T0_C,
            "T_out_outside_C": res_inlet.T_out_outside - T0_C,
            "inside_v_m_s": res_inlet.inside_velocity_mean,
            "outside_v_m_s": res_inlet.outside_velocity_mean,
            "inside_rho_kg_m3": res_inlet.inside_props_mean.rho,
            "outside_rho_kg_m3": res_inlet.outside_props_mean.rho,
            "inside_mu_Pa_s": res_inlet.inside_props_mean.mu,
            "outside_mu_Pa_s": res_inlet.outside_props_mean.mu,
            "inside_k_W_mK": res_inlet.inside_props_mean.k,
            "outside_k_W_mK": res_inlet.outside_props_mean.k,
            "inside_cp_J_kgK": res_inlet.inside_props_mean.cp,
            "outside_cp_J_kgK": res_inlet.outside_props_mean.cp,
            "inside_molar_mass_kg_mol": inside_molar_mass,
            "outside_molar_mass_kg_mol": outside_molar_mass,
            "inside_Vdot_m3_s": inside.m_dot / res_inlet.inside_props_mean.rho,
            "outside_Vdot_m3_s": outside.m_dot / res_inlet.outside_props_mean.rho,
            "A_inner_m2": bundle.total_inner_area,
            "A_outer_m2": bundle.total_outer_area,
            "inside_dp_Pa": res_inlet.final_result.tube_side_hydraulic.dp_total,
            "inside_dp_friction_Pa": res_inlet.final_result.tube_side_hydraulic.dp_friction,
            "inside_dp_acceleration_Pa": res_inlet.final_result.tube_side_hydraulic.dp_acceleration,
            "inside_dp_bundle_Pa": res_inlet.final_result.tube_side_hydraulic.dp_tube_bundle,
            "inside_dp_local_Pa": res_inlet.inside_dp_local,
            "outside_dp_drag_Pa": res_inlet.outside_dp_drag,
            "outside_dp_acceleration_Pa": res_inlet.outside_dp_acceleration,
            "outside_dp_local_Pa": res_inlet.outside_dp_local,
            "outside_dp_Pa": res_inlet.outside_dp_total,
        },
    ])

    print(f"Mean-property vs. inlet-only duty shift: {duty_shift_pct:+.2f} %")
    display(comparison)
else:
    print(
        "mode='rating' selected -- skipping the inlet-only Simulation "
        "comparison. Set mode='simulation' above to compare mean-property "
        "vs. inlet-only."
    )

mode='rating' selected -- skipping the inlet-only Simulation comparison. Set mode='simulation' above to compare mean-property vs. inlet-only.


## Rating: Closed Heat Balance → Overdesign

Only meaningful for `mode="rating"` (skipped otherwise). Uses the same
geometry and side inputs (`inside`/`outside`) as Simulation above, but
treats the outlet temperatures as *known* (target/measured), closing the
heat balance via `BalanceSideSpec`/`hx.rate(...)` and reporting
`overdesign_factor`/`ua_margin` instead of computing outlet temperatures.

`inside_T_out_target_C`/`outside_T_out_target_C` (Mode Selection above)
default to the `surface_margin=0` reference Simulation's own outlet
temperatures, so an out-of-the-box run starts at `overdesign_factor ≈ 0`.
Edit those two variables and re-run from Mode Selection down to see the
overdesign move (positive = spare surface, negative = shortfall).

In [9]:
if mode == "rating":
    inside_bal = BalanceSideSpec(
        provider=inside.provider, p=inside.p,
        m_dot=inside.m_dot, T_in=inside.T_in,
        T_out=c_to_k(inside_T_out_target_C),
    )
    outside_bal = BalanceSideSpec(
        provider=outside.provider, p=outside.p,
        m_dot=outside.m_dot, T_in=outside.T_in,
        T_out=c_to_k(outside_T_out_target_C),
    )

    res_rating = hx.rate(
        inside_bal, outside_bal, include_simulation=True,
        euler_provider=euler_provider,
    )

    print("=" * 70)
    print("RATING RESULT (this is the requested output for mode='rating')")
    print("=" * 70)

    display(pd.Series({
        "overdesign_factor": res_rating.overdesign_factor,
        "ua_margin": res_rating.ua_margin,
        "A_o_m2": res_rating.A_o,
        "A_required_m2": res_rating.A_required,
        "UA_required_W_K": res_rating.UA_required,
        "UA_actual_W_K": res_rating.UA_actual,
        "U_mean_W_m2K": res_rating.U_mean,
        "Q_required_kW": res_rating.Q_required / 1e3,
        "Q_achievable_kW": (
            res_rating.Q_achievable / 1e3 if res_rating.Q_achievable is not None else None
        ),
        "closed_balance_effectiveness": res_rating.closed_balance.effectiveness,
        "inside_T_out_target_C": inside_T_out_target_C,
        "outside_T_out_target_C": outside_T_out_target_C,
    }, name="value").to_frame())

    if res_rating.warnings:
        print("\nWARNINGS:")
        for w in res_rating.warnings:
            print(f"  [{w.severity}] {w.code}: {w.message}")
    else:
        print("\nNo warnings.")
else:
    print(
        "mode='simulation' selected -- skipping Rating. Set mode='rating' "
        "above (and optionally edit the target outlet temperatures) to "
        "close a heat balance and compute overdesign instead."
    )

RATING RESULT (this is the requested output for mode='rating')


,value
overdesign_factor,0.374926
ua_margin,0.374926
A_o_m2,210.662637
A_required_m2,153.217486
UA_required_W_K,2915.548582
UA_actual_W_K,4008.662251
U_mean_W_m2K,19.028824
Q_required_kW,580.798705
Q_achievable_kW,704.698386
closed_balance_effectiveness,0.388514



WARNINGS:
  [warning] heat_balance_over_specified: heat_balance: a fully specified side implies a duty of 704697 W, which does not match the resolved balance duty of 580799 W within tolerance=0.001.
  [info] tube_bundle_hydraulics_midpoint_temperature_fallback: tube_bundle_hydraulics: complete inlet/outlet enthalpy data are unavailable; using arithmetic temperature midpoint.
  [info] tube_bundle_hydraulics_pass_boundary_temperature_fallback: tube_bundle_hydraulics: complete inlet/outlet enthalpy data are unavailable; using linear temperature interpolation for tube-pass boundary states.
  [info] wall_temperature_envelope_0d_estimate: Wall-temperature minimum and maximum are estimated from a 0D inlet/outlet endpoint envelope. They are not local extrema from a spatially segmented exchanger model.


## Energy-Balance Sanity Check (Reference Simulation)

Checks `q` implied by each side's own `m_dot * cp_mean * dT` against the
reported duty of the converged mean-property reference Simulation
(`res_mean`, always computed above regardless of `mode`).

In [10]:
q_inside = inside.m_dot * res_mean.inside_props_mean.cp * (res_mean.T_out_inside - inside.T_in)
q_outside = outside.m_dot * res_mean.outside_props_mean.cp * (outside.T_in - res_mean.T_out_outside)

print(f"q_inside   = {q_inside / 1e3:.2f} kW")
print(f"q_outside  = {q_outside / 1e3:.2f} kW")
print(f"q (simulate) = {res_mean.q / 1e3:.2f} kW")

assert abs(q_inside - res_mean.q) / res_mean.q < 0.02
assert abs(q_outside - res_mean.q) / res_mean.q < 0.02

if res_mean.warnings:
    print("\nWARNINGS:")
    for w in res_mean.warnings:
        print(f"  [{w.severity}] {w.code}: {w.message}")
else:
    print("\nNo warnings.")

q_inside   = 704.70 kW
q_outside  = 704.70 kW
q (simulate) = 704.70 kW

WARNINGS:
  [info] tube_bundle_hydraulics_midpoint_temperature_fallback: tube_bundle_hydraulics: complete inlet/outlet enthalpy data are unavailable; using arithmetic temperature midpoint.
  [info] tube_bundle_hydraulics_pass_boundary_temperature_fallback: tube_bundle_hydraulics: complete inlet/outlet enthalpy data are unavailable; using linear temperature interpolation for tube-pass boundary states.
  [info] wall_temperature_envelope_0d_estimate: Wall-temperature minimum and maximum are estimated from a 0D inlet/outlet endpoint envelope. They are not local extrema from a spatially segmented exchanger model.


## Inlet, midpoint, and outlet fluid properties

Point states below come directly from the solver's existing hydraulic results. The wet midpoint uses arithmetic mean temperature and water ratio; no transport-property provider is called for presentation.

In [11]:
import math
import pandas as pd


def endpoint_property_table(solver_result, side):
    """Read solver-owned hydraulic point states without provider calls."""
    states = (
        ("inlet", getattr(solver_result, f"{side}_properties_inlet")),
        ("midpoint", getattr(solver_result, f"{side}_properties_midpoint")),
        ("outlet", getattr(solver_result, f"{side}_properties_outlet")),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "T [°C]": state.T - 273.15,
                "p [Pa]": state.p,
                "rho [kg/m³]": state.rho,
                "cp [J/(kg·K)]": state.cp,
                "mu [Pa·s]": state.mu,
                "k [W/(m·K)]": state.k,
                "Pr [-]": state.Pr,
            }
            for name, state in states
            if state is not None
        ]
    ).set_index("state")


def representative_0d_property_table(solver_result):
    """Keep representative thermal properties separate from point states."""
    if hasattr(solver_result, "inside_props_mean"):
        pairs = (
            ("inside", solver_result.T_mean_inside, solver_result.inside_props_mean),
            ("outside", solver_result.T_mean_outside, solver_result.outside_props_mean),
        )
    elif getattr(solver_result, "thermal_state", None) is not None:
        thermal = solver_result.thermal_state
        pairs = (
            ("inside", thermal.inside_bulk_temperature, thermal.inside_bulk_props),
            ("outside", thermal.outside_bulk_temperature, thermal.outside_bulk_props),
        )
    else:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "side": side,
                "T representative [°C]": temperature - 273.15,
                "rho [kg/m³]": props.rho,
                "cp [J/(kg·K)]": props.cp,
                "mu [Pa·s]": props.mu,
                "k [W/(m·K)]": props.k,
                "Pr [-]": props.mu * props.cp / props.k,
            }
            for side, temperature, props in pairs
        ]
    ).set_index("side")


def wet_gas_state_table(solver_result, outside_provider_for_result=None):
    """Combine hydraulic states with wet diagnostics already returned by the solver."""
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable or pc.W_in is None:
        return pd.DataFrame()

    W_mid = 0.5 * (pc.W_in + pc.W_out)
    dew_mid = math.nan
    if outside_provider_for_result is not None:
        from core.phase_change.capability import detect_phase_change_capability
        from core.phase_change.integration import _dew_point_at_ratio

        capability = detect_phase_change_capability(outside_provider_for_result)
        midpoint_state = solver_result.outside_properties_midpoint
        dew_mid_value = _dew_point_at_ratio(
            capability, W_mid, p=midpoint_state.p
        )
        dew_mid = math.nan if dew_mid_value is None else dew_mid_value

    dry_flow = pc.m_dot_dry_carrier
    vapor_mid = (
        math.nan
        if dry_flow is None
        else dry_flow * W_mid
    )
    gas_mid = (
        math.nan
        if dry_flow is None
        else dry_flow + vapor_mid
    )
    values = (
        ("inlet", pc.W_in, pc.dew_point_in, pc.m_dot_gas_in, pc.m_dot_water_vapor_in),
        ("midpoint", W_mid, dew_mid, gas_mid, vapor_mid),
        ("outlet", pc.W_out, pc.dew_point_out, pc.m_dot_gas_out, pc.m_dot_water_vapor_out),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "W [kg/kg dry]": W,
                "dew point [°C]": (
                    math.nan if dew_point is None else dew_point - 273.15
                ),
                "m_dot gas [kg/s]": gas_flow,
                "m_dot water vapor [kg/s]": vapor_flow,
            }
            for name, W, dew_point, gas_flow, vapor_flow in values
        ]
    ).set_index("state")


def condensation_summary_table(solver_result):
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable:
        return pd.DataFrame()

    def _to_c(value):
        return math.nan if value is None else value - 273.15

    W_bulk = (
        math.nan
        if pc.W_in is None or pc.W_out is None
        else 0.5 * (pc.W_in + pc.W_out)
    )
    return pd.DataFrame(
        [
            {
                "T_dew_in [degC]": _to_c(pc.dew_point_in),
                "T_dew_out [degC]": _to_c(pc.dew_point_out),
                "T_wall_min [degC]": _to_c(pc.wall_temperature_min),
                "T_wall_mean [degC]": _to_c(pc.wall_temperature_mean),
                "T_wall_max [degC]": _to_c(pc.wall_temperature_max),
                "T_wall_wet_mean [degC]": _to_c(pc.wall_temperature_wet_mean),
                "wet_surface_fraction [-]": pc.wet_surface_fraction,
                "A_wet [m2]": pc.wet_area,
                "A_outside [m2]": pc.outside_total_area,
                "W_bulk representative [kg/kg dry]": W_bulk,
                "W_sat_wet_surface [kg/kg dry]": pc.W_sat_wet_surface,
                "W_in [kg/kg dry]": pc.W_in,
                "W_out [kg/kg dry]": pc.W_out,
                "m_dot condensate [kg/s]": pc.m_dot_condensate,
                "Q_sensible [W]": pc.Q_sensible,
                "Q_latent [W]": pc.Q_latent,
                "Q_total [W]": pc.Q_total,
                "alfa_dry [W/(m2 K)]": pc.alfa_dry,
                "alfa_effective [W/(m2 K)]": pc.alfa_effective,
            }
        ],
        index=["outside"],
    )

In [12]:
endpoint_results = [('Reference Simulation', res_mean)] + ([('Rating', res_rating)] if 'res_rating' in globals() else [])
outside_provider_for_endpoint_table = outside.provider

for result_label, endpoint_result in endpoint_results:
    print(result_label)
    print("Inside")
    display(endpoint_property_table(endpoint_result, "inside"))
    print("Outside")
    display(endpoint_property_table(endpoint_result, "outside"))

    wet_table = wet_gas_state_table(
        endpoint_result, outside_provider_for_endpoint_table
    )
    if not wet_table.empty:
        print("Outside wet-gas mass and dew-point diagnostics")
        display(wet_table)
        display(condensation_summary_table(endpoint_result))

Reference Simulation
Inside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,30.000000,101325.0,1.164734,1006.492185,0.000019,0.026618,0.706669
midpoint,99.689194,101325.0,0.946659,1011.202966,0.000022,0.031598,0.700288
outlet,169.378387,101325.0,0.797448,1019.950203,0.000025,0.036274,0.697934


Outside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,326.000000,101325.0,0.551129,1161.057218,0.000029,0.045537,0.735656
midpoint,286.799612,101325.0,0.589727,1150.533137,0.000027,0.042933,0.734021
outlet,247.247337,101325.0,0.634575,1140.528634,0.000026,0.040286,0.732595


Outside wet-gas mass and dew-point diagnostics


...

Outside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,326.000000,101325.0,0.551129,1161.057218,0.000029,0.045537,0.735656
midpoint,286.799612,101325.0,0.589727,1150.533137,0.000027,0.042933,0.734021
outlet,247.247337,101325.0,0.634575,1140.528634,0.000026,0.040286,0.732595


Outside wet-gas mass and dew-point diagnostics

,W [kg/kg dry],dew point [°C],m_dot gas [kg/s],m_dot water vapor [kg/s]
state,,,,
inlet,0.120005,55.74619,7.777778,0.833366
midpoint,0.120005,55.74619,7.777778,0.833366
outlet,0.120005,55.74619,7.777778,0.833366


,T_dew_in [degC],T_dew_out [degC],T_wall_min [degC],T_wall_mean [degC],T_wall_max [degC],T_wall_wet_mean [degC],wet_surface_fraction [-],A_wet [m2],A_outside [m2],W_bulk representative [kg/kg dry],W_sat_wet_surface [kg/kg dry],W_in [kg/kg dry],W_out [kg/kg dry],m_dot condensate [kg/s],Q_sensible [W],Q_latent [W],Q_total [W],alfa_dry [W/(m2 K)],alfa_effective [W/(m2 K)]
outside,55.74619,55.74619,209.409945,252.241679,292.266739,NaN,None,None,None,0.120005,None,0.120005,0.120005,0.0,0.0,0.0,0.0,None,None


## Representative 0D properties used by the solver

These lumped thermal-model properties are retained separately; they are not substitutes for inlet or outlet states.

In [13]:
for result_label, endpoint_result in endpoint_results:
    representative_table = representative_0d_property_table(endpoint_result)
    if not representative_table.empty:
        print(result_label)
        display(representative_table)

Reference Simulation


,T representative [°C],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
side,,,,,,
inside,99.689194,0.946689,1011.201807,0.000022,0.031598,0.700289
outside,286.623669,0.589904,1150.489076,0.000027,0.042922,0.734014


Rating


,T representative [°C],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
side,,,,,,
inside,99.677229,0.946689,1011.201807,0.000022,0.031598,0.700289
outside,286.631173,0.589904,1150.489076,0.000027,0.042922,0.734014
